# House Prices - Improved Version: Clean Baseline

This notebook is a cleaner learning version of your current work.

The goal is not to jump straight to the best model. The goal is to build a simple, logical workflow that you can reuse in future machine learning projects.

## What you are already doing well

You are already doing several important things correctly:

1. You inspect the data before modeling.
2. You check missing values instead of ignoring them.
3. You use the data description to understand what missing values mean.
4. You compare models using validation scores.
5. You are experimenting and asking why the score changes.

That is the right learning mindset. This notebook just makes the order cleaner.

## Step 1 - Import the basic libraries

Start with only the libraries needed for loading data and measuring a simple baseline.

Keeping the first step small helps you understand what each part is doing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## Step 2 - Load the training data

The code below first tries to load `train.csv` from this local folder.

If you later run the notebook on Kaggle, the second path can be used instead.

In [ ]:
try:
    train = pd.read_csv("train.csv")
except FileNotFoundError:
    train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")

train.shape

## Step 3 - Take a first look

Before cleaning or modeling, check what the data looks like.

This is where you confirm the number of rows, columns, column names, and the target column.

In [ ]:
train.head()

In [ ]:
train.info()

## Step 4 - Look at the target: SalePrice

`SalePrice` is what we want to predict.

Before creating any model, understand the target values. This helps you judge whether your model errors are small or large.

In [ ]:
train["SalePrice"].describe()

In [ ]:
average_price = train["SalePrice"].mean()
median_price = train["SalePrice"].median()

print(f"Average SalePrice: {average_price:,.2f}")
print(f"Median SalePrice:  {median_price:,.2f}")

In this dataset, the average sale price is about **180,921**.

This number is useful because it gives us the simplest possible prediction: predict the average price for every house.

## Step 5 - Remove the Id column

`Id` is only a row identifier. It is not a real house feature.

A model should learn from house information like area, quality, rooms, neighborhood, and condition. It should not learn from row numbers.

In [ ]:
train = train.drop("Id", axis=1)

train.head()

## Step 6 - Separate features and target

Machine learning usually separates the data into:

- `X`: the input features used to make predictions
- `y`: the target value we want to predict

Here, `SalePrice` is the target.

In [ ]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

print("Feature table shape:", X.shape)
print("Target shape:", y.shape)

## Step 7 - Create a train/validation split

We do not judge the model on the same rows it learned from.

The training set is used to learn. The validation set is used to test how well the approach works on unseen rows.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])

## Step 8 - Build the simplest baseline: average prediction

A baseline is the simplest complete method that gives us a score.

This baseline does not use any house features yet. It predicts the average training price for every validation house.

This tells us what error we get before doing any real machine learning.

In [ ]:
average_train_price = y_train.mean()

baseline_predictions = np.full(shape=len(y_val), fill_value=average_train_price)

baseline_mae = mean_absolute_error(y_val, baseline_predictions)
baseline_mse = mean_squared_error(y_val, baseline_predictions)
baseline_rmse = baseline_mse ** 0.5
baseline_r2 = r2_score(y_val, baseline_predictions)

print(f"Average price used for prediction: {average_train_price:,.2f}")
print(f"Baseline MAE:  {baseline_mae:,.2f}")
print(f"Baseline RMSE: {baseline_rmse:,.2f}")
print(f"Baseline R2:   {baseline_r2:.4f}")

## What this baseline means

This average-price baseline is intentionally simple.

Any real model should beat this. If a model cannot beat this, then something is wrong with the model setup, preprocessing, or evaluation.

Next step after this notebook section: create a simple machine learning baseline with basic preprocessing.

## Step 9 - Explore the SalePrice distribution

Now that we have a simple average-price baseline, we can study the target more carefully.

The target is the value we are trying to predict. Here, the target is `SalePrice`.

We are looking for three things:

1. Is the target centered around one common price range?
2. Are there very expensive houses stretching the distribution?
3. Is the distribution skewed?

For now, we only observe. We do not transform the target yet.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(train["SalePrice"], bins=40, kde=True)
plt.title("SalePrice Distribution")
plt.xlabel("SalePrice")
plt.ylabel("Number of Houses")
plt.show()

## Step 10 - Compare mean, median, and skewness

The plot shows the shape visually. These numbers help us describe that shape.

- The **mean** is the average.
- The **median** is the middle value.
- **Skewness** tells us whether the distribution has a long tail.

If the mean is noticeably higher than the median, that often means a few expensive houses are pulling the average upward.

In [ ]:
saleprice_mean = train["SalePrice"].mean()
saleprice_median = train["SalePrice"].median()
saleprice_skew = train["SalePrice"].skew()

print(f"Mean SalePrice:   {saleprice_mean:,.2f}")
print(f"Median SalePrice: {saleprice_median:,.2f}")
print(f"Skewness:         {saleprice_skew:.3f}")

## How data shape guides modeling decisions

When we say we are studying the **shape of the data**, we are trying to understand how the values are distributed.

This helps us decide what kind of model, metric, and preprocessing may make sense later.

The main indicators are:

### 1. Central tendency

Central tendency tells us where the typical value is.

Common measures:

- **Mean**: the average value
- **Median**: the middle value after sorting

If mean and median are close, the data may be fairly balanced.

If the mean is much higher than the median, large values are pulling the average upward.

In our dataset:

- Mean `SalePrice` is about **180,921**
- Median `SalePrice` is about **163,000**

This suggests that some expensive houses are pulling the average upward.

### 2. Spread

Spread tells us how far values are from each other.

Common measures:

- **Minimum and maximum**: the smallest and largest values
- **Standard deviation**: how much values usually vary from the mean
- **Interquartile range**: the middle 50% of values

In our dataset:

- Minimum `SalePrice` is **34,900**
- Maximum `SalePrice` is **755,000**

That is a wide spread. This matters because large price differences can make errors look very large, especially with RMSE.

### 3. Skewness

Skewness tells us whether the distribution has a longer tail on one side.

- Skewness near `0`: roughly balanced distribution
- Positive skewness: long tail on the right side
- Negative skewness: long tail on the left side

In our dataset, `SalePrice` skewness is about **1.88**.

This means the target is **right-skewed**. Most houses are in the lower-to-middle price range, while fewer expensive houses stretch the right tail.

### 4. Outliers

Outliers are unusually high or unusually low values.

Outliers are not automatically wrong. In house prices, an expensive house may be a real valid sale.

But outliers can strongly affect:

- Linear models
- MSE
- RMSE
- Mean values

This is why we inspect them before deciding whether to keep, clip, transform, or investigate them.

### 5. Model choice

The shape of the data can guide which model family we try first.

Linear models, such as Linear Regression, often work better when relationships are fairly smooth and close to linear. They can be sensitive to outliers, skewed variables, and features with very different scales.

Tree-based models, such as Random Forest, Gradient Boosting, LightGBM, and TensorFlow Decision Forests, are usually more flexible. They can handle non-linear relationships well and usually do not require feature scaling.

For this house price dataset, tree-based models are a strong first choice because the data contains many mixed feature types and likely non-linear relationships.

### 6. Metric choice

The distribution also affects which error metric is easier to interpret.

- **MAE** measures average absolute error. It is easier to explain: "on average, predictions are off by this many dollars."
- **MSE** squares errors, so large mistakes become much more important.
- **RMSE** is the square root of MSE. It is still in dollars, but it still penalizes large errors strongly.
- **R2** measures how much variation the model explains compared with a simple average prediction.

For skewed house prices, MAE is often easier to understand, while RMSE is useful because it shows when the model makes large mistakes on expensive houses.

In Kaggle competitions, the evaluation metric may be fixed. In learning notebooks, it is useful to report both MAE and RMSE.

### MAE vs RMSE: what is really happening?

Both MAE and RMSE compare predictions with true values, but they react to mistakes differently.

Suppose the real price is `200,000` and the model predicts `180,000`.

The error is:

`200,000 - 180,000 = 20,000`

For MAE, we take the absolute error:

`abs(20,000) = 20,000`

So MAE thinks of this as a `20,000` dollar mistake.

For MSE, we square the error:

`20,000 ** 2 = 400,000,000`

Squaring makes large mistakes grow very fast.

RMSE then takes the square root of the average squared error, so the final number goes back into the original unit: dollars.

This means:

- MAE treats errors more evenly.
- RMSE gives extra weight to large errors.

Example:

Imagine two models make errors on five houses.

Model A errors:

`10k, 10k, 10k, 10k, 10k`

Model B errors:

`0k, 0k, 0k, 0k, 50k`

Both models have the same MAE:

`50k total error / 5 houses = 10k MAE`

But Model B has one very large mistake. RMSE will punish Model B more because the `50k` error gets squared before averaging.

So MAE answers:

> On average, how many dollars wrong are we?

RMSE answers:

> Are we making some very large mistakes?

For our house price dataset, this matters because expensive houses create a long right tail. If the model predicts normal houses well but badly misses expensive houses, RMSE will increase more strongly than MAE.

That is why we read them together:

- If MAE is reasonable but RMSE is much larger, the model may have a few big misses.
- If both MAE and RMSE improve, the model is probably improving overall.
- If RMSE improves but MAE does not improve much, the model may mainly be reducing large mistakes.
- If MAE improves but RMSE stays high, the model may be better on typical houses but still weak on expensive or unusual houses.

### 7. Preprocessing choices

The distribution helps us decide preprocessing steps.

- **Scaling**: often useful for linear models, Ridge, Lasso, KNN, SVM, and neural networks. Usually not needed for tree-based models.
- **Log transform**: useful when a target or feature is strongly right-skewed. It compresses very large values and can make patterns easier for some models.
- **Clipping**: limits extreme values. This can help if outliers are noisy or unrealistic, but it can remove real information if done carelessly.
- **Feature engineering**: creates more meaningful variables, such as total square footage, house age, or total bathrooms.

For our current `SalePrice` target, the right skew suggests that a later experiment with `log1p(SalePrice)` is reasonable.

But we should not transform immediately. First we create a clean baseline, then test one improvement at a time.

### What our dataset is telling us so far

From the current `SalePrice` analysis:

- The target is right-skewed.
- The mean is higher than the median.
- Expensive houses stretch the distribution to the right.
- MAE will be easier to interpret in dollars.
- RMSE may be larger because it punishes big errors more.
- A log transform may be worth testing later.
- Tree-based models are likely a good first serious model family.

The important habit is: observe first, make one change, measure the result, then decide the next change.

## Teacher checkpoint

Pause here before changing the data.

Look at the plot and the mean/median/skewness values. Try to explain in your own words what the distribution is telling us.

Questions to answer:

1. Is `SalePrice` evenly distributed, or is it skewed?
2. Is the mean higher or lower than the median?
3. What does that suggest about expensive houses in the dataset?

After that, we can decide whether a log transformation should be tested later.

## Step 11 - First real ML baseline

Now we create the first real machine learning baseline.

This model uses the actual house features, but keeps preprocessing simple.

We are still not doing feature engineering, log transformation, or tuning.

The goal is to answer one question:

> Can a basic model using house features beat the average-price baseline?

This pipeline does three things:

1. Fills missing numeric values with the median.
2. Fills missing categorical values with `None`.
3. One-hot encodes categorical columns.

Then it trains a `RandomForestRegressor`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

The model below is intentionally plain.

We use `random_state=42` so the result is repeatable.

We use `n_estimators=200`, meaning the random forest builds 200 trees and averages their predictions.

In [ ]:
rf_baseline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_baseline.fit(X_train, y_train)

In [ ]:
rf_predictions = rf_baseline.predict(X_val)

rf_mae = mean_absolute_error(y_val, rf_predictions)
rf_mse = mean_squared_error(y_val, rf_predictions)
rf_rmse = rf_mse ** 0.5
rf_r2 = r2_score(y_val, rf_predictions)

print(f"Random Forest baseline MAE:  {rf_mae:,.2f}")
print(f"Random Forest baseline RMSE: {rf_rmse:,.2f}")
print(f"Random Forest baseline R2:   {rf_r2:.4f}")

## Step 12 - Compare both baselines

Now compare the dummy baseline with the first real ML baseline.

The real model should have lower MAE and lower RMSE than the average-price baseline.

If it does, then the house features are helping the model predict price.

In [ ]:
baseline_comparison = pd.DataFrame({
    "model": ["Average price baseline", "Random Forest baseline"],
    "MAE": [baseline_mae, rf_mae],
    "RMSE": [baseline_rmse, rf_rmse],
    "R2": [baseline_r2, rf_r2]
})

baseline_comparison

## Step 13 - Interpret the ML baseline

Before improving the model, pause and interpret what happened.

We want to answer these questions:

1. Did the ML model beat the average baseline?
2. By how much did MAE improve?
3. By how much did RMSE improve?
4. Is RMSE still much higher than MAE?
5. What does that suggest about big errors?

This step is important because model improvement should be based on evidence, not guessing.

In [ ]:
mae_improvement = baseline_mae - rf_mae
rmse_improvement = baseline_rmse - rf_rmse

mae_improvement_pct = mae_improvement / baseline_mae * 100
rmse_improvement_pct = rmse_improvement / baseline_rmse * 100

print("1. Did the ML model beat the average baseline?")
print("   Yes, if Random Forest MAE and RMSE are lower than the average baseline.")
print()

print("2. By how much did MAE improve?")
print(f"   MAE improved by {mae_improvement:,.2f} dollars ({mae_improvement_pct:.1f}%).")
print()

print("3. By how much did RMSE improve?")
print(f"   RMSE improved by {rmse_improvement:,.2f} dollars ({rmse_improvement_pct:.1f}%).")
print()

print("4. Is RMSE still much higher than MAE?")
print(f"   MAE:  {rf_mae:,.2f}")
print(f"   RMSE: {rf_rmse:,.2f}")
print()

print("5. What does that suggest about big errors?")
print("   If RMSE is much higher than MAE, the model may still have some large mistakes.")
print("   In this dataset, those large mistakes may come from expensive or unusual houses.")

## Step 14 - Test log-transforming SalePrice

Now we test our first improvement idea.

We saw earlier that `SalePrice` is right-skewed. A few expensive houses stretch the target distribution to the right.

A log transformation compresses large values more than small values. This can make the target easier for the model to learn.

Important: we keep the same features, same preprocessing, and same model type.

The only change is the target:

- Train on `log1p(SalePrice)`
- Predict logged prices
- Convert predictions back to dollars using `expm1`
- Evaluate MAE and RMSE in dollars

This keeps the experiment fair.

In [ ]:
y_train_log = np.log1p(y_train)

rf_log_target = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_log_target.fit(X_train, y_train_log)

In [ ]:
log_predictions = rf_log_target.predict(X_val)
rf_log_predictions = np.expm1(log_predictions)

rf_log_mae = mean_absolute_error(y_val, rf_log_predictions)
rf_log_mse = mean_squared_error(y_val, rf_log_predictions)
rf_log_rmse = rf_log_mse ** 0.5
rf_log_r2 = r2_score(y_val, rf_log_predictions)

print(f"Random Forest with log target MAE:  {rf_log_mae:,.2f}")
print(f"Random Forest with log target RMSE: {rf_log_rmse:,.2f}")
print(f"Random Forest with log target R2:   {rf_log_r2:.4f}")

## Step 15 - Compare normal target vs log target

Now compare the same model with and without the log-transformed target.

If the log target improves MAE and RMSE, it suggests the skewed target was making the original prediction task harder.

If it does not improve the score, that is also useful. It means this model may already handle the skew reasonably well, or that a different improvement should be tested next.

In [ ]:
log_comparison = pd.DataFrame({
    "model": ["Average price baseline", "Random Forest baseline", "Random Forest log target"],
    "MAE": [baseline_mae, rf_mae, rf_log_mae],
    "RMSE": [baseline_rmse, rf_rmse, rf_log_rmse],
    "R2": [baseline_r2, rf_r2, rf_log_r2]
})

log_comparison

## Step 16 - Improve feature handling: ordinal quality columns

The log-target experiment was mixed, so we will not make it the new main baseline yet.

The next improvement is to use the meaning of the columns more carefully.

In the Kaggle data description, many features are not just random categories. They have a natural order.

For example:

- `Ex` means Excellent
- `Gd` means Good
- `TA` means Typical/Average
- `Fa` means Fair
- `Po` means Poor
- `None` means the feature does not exist, such as no garage, no basement, no fireplace, or no pool

One-hot encoding treats these as unrelated labels. Ordinal encoding tells the model the order:

`None < Po < Fa < TA < Gd < Ex`

This is still a simple improvement. We are not creating new features yet. We are only representing existing features more intelligently.

In [ ]:
quality_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

quality_columns = [
    "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual", "GarageCond",
    "PoolQC"
]

bsmt_exposure_map = {
    "None": 0,
    "No": 1,
    "Mn": 2,
    "Av": 3,
    "Gd": 4
}

bsmt_fin_type_map = {
    "None": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
}

garage_finish_map = {
    "None": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
}

Now we copy the training and validation feature tables, then apply the fixed mappings.

These mappings come from the data description, not from the validation target, so this does not leak information from validation into training.

In [ ]:
X_train_ordinal = X_train.copy()
X_val_ordinal = X_val.copy()

for col in quality_columns:
    X_train_ordinal[col] = X_train_ordinal[col].fillna("None").map(quality_map)
    X_val_ordinal[col] = X_val_ordinal[col].fillna("None").map(quality_map)

for col in ["BsmtExposure"]:
    X_train_ordinal[col] = X_train_ordinal[col].fillna("None").map(bsmt_exposure_map)
    X_val_ordinal[col] = X_val_ordinal[col].fillna("None").map(bsmt_exposure_map)

for col in ["BsmtFinType1", "BsmtFinType2"]:
    X_train_ordinal[col] = X_train_ordinal[col].fillna("None").map(bsmt_fin_type_map)
    X_val_ordinal[col] = X_val_ordinal[col].fillna("None").map(bsmt_fin_type_map)

for col in ["GarageFinish"]:
    X_train_ordinal[col] = X_train_ordinal[col].fillna("None").map(garage_finish_map)
    X_val_ordinal[col] = X_val_ordinal[col].fillna("None").map(garage_finish_map)

After mapping these ordered columns to numbers, we rebuild the preprocessing pipeline.

The remaining text columns still need one-hot encoding. The newly mapped quality columns are now numeric and will be handled with the numeric columns.

In [ ]:
numeric_features_ordinal = X_train_ordinal.select_dtypes(include=["int64", "float64"]).columns
categorical_features_ordinal = X_train_ordinal.select_dtypes(include=["object"]).columns

numeric_transformer_ordinal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_ordinal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_ordinal = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_ordinal, numeric_features_ordinal),
        ("cat", categorical_transformer_ordinal, categorical_features_ordinal)
    ]
)

In [ ]:
rf_ordinal = Pipeline(steps=[
    ("preprocessor", preprocessor_ordinal),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_ordinal.fit(X_train_ordinal, y_train)

rf_ordinal_predictions = rf_ordinal.predict(X_val_ordinal)

rf_ordinal_mae = mean_absolute_error(y_val, rf_ordinal_predictions)
rf_ordinal_mse = mean_squared_error(y_val, rf_ordinal_predictions)
rf_ordinal_rmse = rf_ordinal_mse ** 0.5
rf_ordinal_r2 = r2_score(y_val, rf_ordinal_predictions)

print(f"Random Forest with ordinal quality features MAE:  {rf_ordinal_mae:,.2f}")
print(f"Random Forest with ordinal quality features RMSE: {rf_ordinal_rmse:,.2f}")
print(f"Random Forest with ordinal quality features R2:   {rf_ordinal_r2:.4f}")

## Step 17 - Compare feature handling experiments

Now compare the original Random Forest baseline with the ordinal-feature version.

Decision rule:

- If MAE and RMSE both improve, ordinal encoding is helping.
- If MAE improves but RMSE gets worse, it may help typical houses but hurt large-error cases.
- If scores are almost the same, Random Forest may already be handling the one-hot version well.
- If both get worse, keep the simpler baseline for now.

The goal is not to force every idea to work. The goal is to test one idea clearly.

In [ ]:
feature_handling_comparison = pd.DataFrame({
    "model": [
        "Average price baseline",
        "Random Forest baseline",
        "Random Forest log target",
        "Random Forest ordinal features"
    ],
    "MAE": [baseline_mae, rf_mae, rf_log_mae, rf_ordinal_mae],
    "RMSE": [baseline_rmse, rf_rmse, rf_log_rmse, rf_ordinal_rmse],
    "R2": [baseline_r2, rf_r2, rf_log_r2, rf_ordinal_r2]
})

feature_handling_comparison

## Step 18 - Feature engineering

The ordinal encoding experiment did not clearly improve the Random Forest baseline.

So now we try feature engineering.

Feature engineering means creating new columns from existing columns so the model can see useful patterns more directly.

We will keep this beginner-friendly and create features that make real-world sense:

- `TotalSF`: total square footage across basement, first floor, and second floor
- `TotalBathrooms`: total bathrooms, counting half bathrooms as `0.5`
- `HouseAge`: how old the house was when sold
- `RemodAge`: how long since the house was remodeled when sold
- `TotalPorchSF`: total porch/deck area
- `HasGarage`: whether the house has garage space
- `HasBasement`: whether the house has basement space
- `HasFireplace`: whether the house has at least one fireplace
- `HasPool`: whether the house has pool area

Important: we create the same features for training and validation using only existing feature columns. We are not using `SalePrice` to create these features.

In [ ]:
def add_house_features(df):
    df = df.copy()
    
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    
    df["TotalBathrooms"] = (
        df["FullBath"]
        + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"]
        + 0.5 * df["BsmtHalfBath"]
    )
    
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    
    df["TotalPorchSF"] = (
        df["WoodDeckSF"]
        + df["OpenPorchSF"]
        + df["EnclosedPorch"]
        + df["3SsnPorch"]
        + df["ScreenPorch"]
    )
    
    df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
    df["HasBasement"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)
    df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype(int)
    
    return df

X_train_fe = add_house_features(X_train)
X_val_fe = add_house_features(X_val)

print("Original feature count:", X_train.shape[1])
print("Feature-engineered count:", X_train_fe.shape[1])

Now we train the same Random Forest model again.

The only difference is that the model now receives the extra engineered features.

We keep the same basic preprocessing: median for numeric missing values and `None` plus one-hot encoding for categorical missing values.

In [ ]:
numeric_features_fe = X_train_fe.select_dtypes(include=["int64", "float64"]).columns
categorical_features_fe = X_train_fe.select_dtypes(include=["object"]).columns

numeric_transformer_fe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_fe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_fe = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_fe, numeric_features_fe),
        ("cat", categorical_transformer_fe, categorical_features_fe)
    ]
)

In [ ]:
rf_feature_engineered = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_feature_engineered.fit(X_train_fe, y_train)

rf_fe_predictions = rf_feature_engineered.predict(X_val_fe)

rf_fe_mae = mean_absolute_error(y_val, rf_fe_predictions)
rf_fe_mse = mean_squared_error(y_val, rf_fe_predictions)
rf_fe_rmse = rf_fe_mse ** 0.5
rf_fe_r2 = r2_score(y_val, rf_fe_predictions)

print(f"Random Forest with feature engineering MAE:  {rf_fe_mae:,.2f}")
print(f"Random Forest with feature engineering RMSE: {rf_fe_rmse:,.2f}")
print(f"Random Forest with feature engineering R2:   {rf_fe_r2:.4f}")

## Step 19 - Compare feature engineering results

Now compare the feature-engineered model against previous experiments.

Decision rule:

- If MAE and RMSE both improve, the engineered features are useful.
- If only MAE improves, the features may help typical houses but not large-error cases.
- If only RMSE improves, the features may help reduce large mistakes.
- If both get worse, remove or rethink these engineered features.

Remember: feature engineering is an experiment. We keep the features only if the validation results support them.

In [ ]:
final_comparison = pd.DataFrame({
    "model": [
        "Average price baseline",
        "Random Forest baseline",
        "Random Forest log target",
        "Random Forest ordinal features",
        "Random Forest feature engineered"
    ],
    "MAE": [baseline_mae, rf_mae, rf_log_mae, rf_ordinal_mae, rf_fe_mae],
    "RMSE": [baseline_rmse, rf_rmse, rf_log_rmse, rf_ordinal_rmse, rf_fe_rmse],
    "R2": [baseline_r2, rf_r2, rf_log_r2, rf_ordinal_r2, rf_fe_r2]
})

final_comparison

## Step 20 - Try a stronger model: Gradient Boosting

Feature engineering improved the Random Forest model, so we keep those engineered features.

Now we test a different model while keeping the same feature-engineered data and preprocessing.

This is important: we only change one thing at a time.

Gradient Boosting often works well on tabular regression problems because it builds trees sequentially. Each new tree tries to correct mistakes made by the previous trees.

Decision rule:

- If Gradient Boosting improves MAE and RMSE, it becomes the new best model.
- If it does not improve, we keep Random Forest with feature engineering as the better simple model.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
gb_feature_engineered = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("model", GradientBoostingRegressor(random_state=42))
])

gb_feature_engineered.fit(X_train_fe, y_train)

gb_fe_predictions = gb_feature_engineered.predict(X_val_fe)

gb_fe_mae = mean_absolute_error(y_val, gb_fe_predictions)
gb_fe_mse = mean_squared_error(y_val, gb_fe_predictions)
gb_fe_rmse = gb_fe_mse ** 0.5
gb_fe_r2 = r2_score(y_val, gb_fe_predictions)

print(f"Gradient Boosting with feature engineering MAE:  {gb_fe_mae:,.2f}")
print(f"Gradient Boosting with feature engineering RMSE: {gb_fe_rmse:,.2f}")
print(f"Gradient Boosting with feature engineering R2:   {gb_fe_r2:.4f}")

## Step 21 - Compare model families

Now we compare model families using the same feature-engineered data.

This tells us whether changing the model improved performance more than changing the features.

In [ ]:
model_family_comparison = pd.DataFrame({
    "model": [
        "Average price baseline",
        "Random Forest baseline",
        "Random Forest feature engineered",
        "Gradient Boosting feature engineered"
    ],
    "MAE": [baseline_mae, rf_mae, rf_fe_mae, gb_fe_mae],
    "RMSE": [baseline_rmse, rf_rmse, rf_fe_rmse, gb_fe_rmse],
    "R2": [baseline_r2, rf_r2, rf_fe_r2, gb_fe_r2]
})

model_family_comparison

## Step 22 - Try another tree model: Extra Trees

Next we try `ExtraTreesRegressor`.

Extra Trees is similar to Random Forest, but it adds more randomness when choosing tree splits.

Why try it here?

- It works well with tabular data.
- It does not require scaling.
- It can handle the same preprocessed feature-engineered data.
- It gives us another tree-based comparison before moving to external libraries like LightGBM.

Again, we keep the same feature-engineered data and preprocessing. Only the model changes.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

In [ ]:
extra_trees_feature_engineered = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("model", ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

extra_trees_feature_engineered.fit(X_train_fe, y_train)

extra_trees_predictions = extra_trees_feature_engineered.predict(X_val_fe)

extra_trees_mae = mean_absolute_error(y_val, extra_trees_predictions)
extra_trees_mse = mean_squared_error(y_val, extra_trees_predictions)
extra_trees_rmse = extra_trees_mse ** 0.5
extra_trees_r2 = r2_score(y_val, extra_trees_predictions)

print(f"Extra Trees with feature engineering MAE:  {extra_trees_mae:,.2f}")
print(f"Extra Trees with feature engineering RMSE: {extra_trees_rmse:,.2f}")
print(f"Extra Trees with feature engineering R2:   {extra_trees_r2:.4f}")

## Step 23 - Compare tree model variants

Now compare Random Forest, Gradient Boosting, and Extra Trees on the same feature-engineered data.

Decision rule:

- Choose the model with the best validation performance overall.
- If one model has slightly better MAE but worse RMSE, inspect whether we care more about typical errors or large errors.
- If the differences are small, prefer the simpler and more stable model.

At this stage, we are still learning. The goal is not only the best score, but understanding why each change helps or does not help.

In [ ]:
tree_model_comparison = pd.DataFrame({
    "model": [
        "Average price baseline",
        "Random Forest feature engineered",
        "Gradient Boosting feature engineered",
        "Extra Trees feature engineered"
    ],
    "MAE": [baseline_mae, rf_fe_mae, gb_fe_mae, extra_trees_mae],
    "RMSE": [baseline_rmse, rf_fe_rmse, gb_fe_rmse, extra_trees_rmse],
    "R2": [baseline_r2, rf_fe_r2, gb_fe_r2, extra_trees_r2]
})

tree_model_comparison

## Step 24 - Run a wider model comparison

Now that we have a clean feature-engineered dataset, we can compare several good baseline models in one place.

This is the right time to compare models because:

- Missing values are handled consistently.
- Categorical columns are encoded consistently.
- The same train/validation split is used.
- The same engineered features are used.
- Only the model changes.

This makes the comparison fair.

We will try:

- Random Forest
- Extra Trees
- Gradient Boosting
- Hist Gradient Boosting
- Ridge Regression
- Lasso Regression
- Elastic Net

Ridge, Lasso, and Elastic Net are linear models, so they need scaling. The tree models do not need scaling.

Decision rule:

- Use MAE to understand typical dollar error.
- Use RMSE to see whether large mistakes are still happening.
- Use R2 to see how much variation the model explains.
- Prefer a model that improves both MAE and RMSE.
- If scores are very close, prefer the simpler or more stable model.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.preprocessing import StandardScaler

For tree-based models, we keep the same preprocessing used earlier.

For linear models, we add scaling after preprocessing because linear models are sensitive to feature scale.

The code below stores all candidate models in one dictionary so we can loop through them and compare results consistently.

In [ ]:
tree_preprocessor = preprocessor_fe

linear_preprocessor = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("scaler", StandardScaler(with_mean=False))
])

candidate_models = {
    "Random Forest": Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
    
    "Extra Trees": Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
    
    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", GradientBoostingRegressor(random_state=42))
    ]),
    
    "Hist Gradient Boosting": Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", HistGradientBoostingRegressor(random_state=42))
    ]),
    
    "Ridge Regression": Pipeline(steps=[
        ("preprocessor", linear_preprocessor),
        ("model", Ridge(alpha=10.0))
    ]),
    
    "Lasso Regression": Pipeline(steps=[
        ("preprocessor", linear_preprocessor),
        ("model", Lasso(alpha=100.0, max_iter=10000, random_state=42))
    ]),
    
    "Elastic Net": Pipeline(steps=[
        ("preprocessor", linear_preprocessor),
        ("model", ElasticNet(alpha=100.0, l1_ratio=0.5, max_iter=10000, random_state=42))
    ])
}

Now we train each model and collect the same metrics.

This table will become our main evidence for deciding what to try next.

When you run this cell, look for:

- lowest MAE
- lowest RMSE
- highest R2
- whether the same model wins across all metrics

In [ ]:
model_results = []

for model_name, model_pipeline in candidate_models.items():
    model_pipeline.fit(X_train_fe, y_train)
    predictions = model_pipeline.predict(X_val_fe)
    
    mae = mean_absolute_error(y_val, predictions)
    mse = mean_squared_error(y_val, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_val, predictions)
    
    model_results.append({
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

model_results_df = pd.DataFrame(model_results).sort_values(by="RMSE")
model_results_df

## Step 25 - Add optional advanced models

Some strong tabular models are not always installed by default.

If your environment has them, the next two useful models are:

- LightGBM
- XGBoost

These are often very strong on Kaggle tabular problems.

The cells below are optional. If the library is not installed, the cell will print a message and continue.

In [ ]:
optional_results = []

In [ ]:
try:
    from lightgbm import LGBMRegressor
    
    lgbm_model = Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            random_state=42
        ))
    ])
    
    lgbm_model.fit(X_train_fe, y_train)
    lgbm_predictions = lgbm_model.predict(X_val_fe)
    
    optional_results.append({
        "model": "LightGBM",
        "MAE": mean_absolute_error(y_val, lgbm_predictions),
        "RMSE": mean_squared_error(y_val, lgbm_predictions) ** 0.5,
        "R2": r2_score(y_val, lgbm_predictions)
    })
    
except ImportError:
    print("LightGBM is not installed in this environment.")

In [ ]:
try:
    from xgboost import XGBRegressor
    
    xgb_model = Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    xgb_model.fit(X_train_fe, y_train)
    xgb_predictions = xgb_model.predict(X_val_fe)
    
    optional_results.append({
        "model": "XGBoost",
        "MAE": mean_absolute_error(y_val, xgb_predictions),
        "RMSE": mean_squared_error(y_val, xgb_predictions) ** 0.5,
        "R2": r2_score(y_val, xgb_predictions)
    })
    
except ImportError:
    print("XGBoost is not installed in this environment.")

## Step 26 - Final model comparison table

This table combines the required scikit-learn models and any optional advanced models that were available.

Sort by RMSE first because RMSE highlights large mistakes, then also check MAE.

After running this, bring the table back and we will decide the next step together.

In [ ]:
all_results_df = pd.concat(
    [model_results_df, pd.DataFrame(optional_results)],
    ignore_index=True
).sort_values(by="RMSE")

all_results_df

## Step 27 - Try simple ensembling

Ensembling means combining predictions from multiple models.

The idea is simple: different models make different mistakes. If we average their predictions, some mistakes may cancel out.

For this problem, the best expected candidates are usually tree-based models:

- Random Forest
- Extra Trees
- Gradient Boosting
- Hist Gradient Boosting
- LightGBM, if installed
- XGBoost, if installed

We should not automatically assume the ensemble is better. We test it on the validation set just like every other model.

Decision rule:

- If the ensemble improves MAE and RMSE, keep it.
- If it improves only one metric, decide what matters more.
- If it does not improve, keep the best single model.

First, we collect predictions from the strongest always-available tree models.

These models use the same feature-engineered data and the same preprocessing.

In [ ]:
ensemble_predictions = {}

ensemble_base_models = {
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42)
}

for model_name, model in ensemble_base_models.items():
    model_pipeline = Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", model)
    ])
    
    model_pipeline.fit(X_train_fe, y_train)
    ensemble_predictions[model_name] = model_pipeline.predict(X_val_fe)

Now we add optional advanced models if they are installed.

If LightGBM or XGBoost are unavailable, the cell will continue without them.

In [ ]:
try:
    from lightgbm import LGBMRegressor
    
    lgbm_ensemble_model = Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            random_state=42
        ))
    ])
    
    lgbm_ensemble_model.fit(X_train_fe, y_train)
    ensemble_predictions["LightGBM"] = lgbm_ensemble_model.predict(X_val_fe)
    
except ImportError:
    print("LightGBM is not installed, so it will not be included in the ensemble.")

In [ ]:
try:
    from xgboost import XGBRegressor
    
    xgb_ensemble_model = Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    xgb_ensemble_model.fit(X_train_fe, y_train)
    ensemble_predictions["XGBoost"] = xgb_ensemble_model.predict(X_val_fe)
    
except ImportError:
    print("XGBoost is not installed, so it will not be included in the ensemble.")

## Step 28 - Evaluate ensemble predictions

We will try two simple ensemble strategies:

1. **Simple average ensemble**: average all available model predictions equally.
2. **Weighted ensemble**: give slightly more weight to models that are often strong on tabular data.

The weighted ensemble is only a starting point. Later, weights can be tuned using cross-validation.

In [ ]:
prediction_table = pd.DataFrame(ensemble_predictions)

simple_ensemble_predictions = prediction_table.mean(axis=1)

simple_ensemble_mae = mean_absolute_error(y_val, simple_ensemble_predictions)
simple_ensemble_rmse = mean_squared_error(y_val, simple_ensemble_predictions) ** 0.5
simple_ensemble_r2 = r2_score(y_val, simple_ensemble_predictions)

print(f"Simple ensemble MAE:  {simple_ensemble_mae:,.2f}")
print(f"Simple ensemble RMSE: {simple_ensemble_rmse:,.2f}")
print(f"Simple ensemble R2:   {simple_ensemble_r2:.4f}")

In [ ]:
weights = {}

for model_name in prediction_table.columns:
    if model_name in ["LightGBM", "XGBoost", "Gradient Boosting", "Hist Gradient Boosting"]:
        weights[model_name] = 2.0
    else:
        weights[model_name] = 1.0

weight_values = np.array([weights[col] for col in prediction_table.columns])
weighted_ensemble_predictions = np.average(
    prediction_table.values,
    axis=1,
    weights=weight_values
)

weighted_ensemble_mae = mean_absolute_error(y_val, weighted_ensemble_predictions)
weighted_ensemble_rmse = mean_squared_error(y_val, weighted_ensemble_predictions) ** 0.5
weighted_ensemble_r2 = r2_score(y_val, weighted_ensemble_predictions)

print("Models included:", list(prediction_table.columns))
print("Weights used:", weights)
print()
print(f"Weighted ensemble MAE:  {weighted_ensemble_mae:,.2f}")
print(f"Weighted ensemble RMSE: {weighted_ensemble_rmse:,.2f}")
print(f"Weighted ensemble R2:   {weighted_ensemble_r2:.4f}")

## Step 29 - Compare individual models vs ensembles

This final table lets us compare:

- the best individual models
- the simple average ensemble
- the weighted ensemble

If an ensemble wins, it may be a good candidate for Kaggle submission.

If a single model wins, keep the single model. Simpler is better when performance is similar.

In [ ]:
ensemble_rows = pd.DataFrame([
    {
        "model": "Simple average ensemble",
        "MAE": simple_ensemble_mae,
        "RMSE": simple_ensemble_rmse,
        "R2": simple_ensemble_r2
    },
    {
        "model": "Weighted ensemble",
        "MAE": weighted_ensemble_mae,
        "RMSE": weighted_ensemble_rmse,
        "R2": weighted_ensemble_r2
    }
])

final_model_and_ensemble_comparison = pd.concat(
    [all_results_df, ensemble_rows],
    ignore_index=True
).sort_values(by="RMSE")

final_model_and_ensemble_comparison